# MIROC6

Worked fine on re-run.

In [1]:
import xarray as xr
import os
import numpy as np
from eofs.standard import Eof
import logging

from functions.SIT_functions.SIT_eddy_feedback_functions import eof_calc, vert_integrate, read_daily_averages, propagate_missing_data_to_all_vars

/home/users/cturrell/miniforge3/envs/eddy/lib/python3.10/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
exp_type = 'cmip'
output_eof_file = '/home/users/cturrell/documents/eddy_feedback/b-parameter/cmip6_b/hist_NaN_issues/data'
force_eof_recalculate = True
pfull_slice = slice(850., 100.)
subtract_annual_cycle=True
eof_vars = ['ucomp', 'div1_QG', 'div1_QG_123', 'div1_QG_gt3']
n_eofs = 3
season_month_dict = {'DJF':[12,1,2,]}
propogate_all_nans = True

In [3]:
miroc_path = '/gws/ssde/j25a/arctic_connect/cturrell/CMIP6/historical/MIROC6/1850_2015/6hrPlevPt/'
anom_ds_miroc = xr.open_dataset(os.path.join(miroc_path, '1979_2015', 'anoms.nc'))
dataset_miroc = read_daily_averages(os.path.join(miroc_path, 'yearly_data'), start_month=1979, end_month=2015, exp_type='cmip6')

eof_miroc = eof_calc('cmip', output_eof_file, force_eof_recalculate, dataset_miroc, pfull_slice,
                              subtract_annual_cycle, eof_vars, n_eofs, season_month_dict, anom_ds_miroc,
                              propogate_all_nans, level_subset=[250., 500., 850.],
                              pressure_weighted=True)

2026-04-15 11:47:11,769 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1135) >> reading daily average files
2026-04-15 11:47:11,769 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1135) >> reading daily average files


2026-04-15 11:47:14,499 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1176) >> finished reading daily average files
2026-04-15 11:47:14,499 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1176) >> finished reading daily average files
2026-04-15 11:47:14,521 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1201) >> duplicate times are present!!!
2026-04-15 11:47:14,521 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1201) >> duplicate times are present!!!
2026-04-15 11:47:14,820 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1205) >> duplicate times removed. Was 13185 now 13150
2026-04-15 11:47:14,820 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1205) >> duplicate times removed. Was 13185 now 13150
2026-04-15 11:47:14,842 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1208) >> duplicate times are NOT present - continuing
2026-04-15 11:47:14,842 [INFO]: SIT_eddy_feedback_funct

In [4]:
def obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds,
             propagate_all_nans, level_subset=None):  

    pfull_selector = level_subset if level_subset is not None else pfull_slice

    # Resolve level_subset to nearest actual levels in the data to avoid
    # exact-match failures on models with slightly different pressure coordinates
    if isinstance(pfull_selector, list):
        pfull_selector = [float(anom_ds['ucomp_anom'].sel(pfull=lev, method='nearest').pfull.values)
                          for lev in pfull_selector]

    if np.all(dataset.lat.diff('lat')<0.):
        hemisphere_slice_dict = {'n':slice(80.,10.),
                                's':slice(-10., -80.),                          
                                }
    else:
        hemisphere_slice_dict = {'n':slice(10.,80.),
                                's':slice(-80., -10.),                             
                                }        

    if propagate_all_nans:
        output_eof_file_use = output_eof_file.split('.nc')[0]+'_prop_nans.nc'
    else:
        output_eof_file_use = output_eof_file

    if os.path.isfile(output_eof_file_use) and not force_eof_recalculate:
        logging.info('attempting to read in EOF data')
        eof_ds = xr.open_mfdataset(output_eof_file_use, decode_times=True)
        logging.info('SUCCESS')
    else:
        logging.info('failed to read in previously calculated EOF data - CALCULATING')

        logging.info('calculating anomalies')
        if exp_type=='held_suarez':
            u_anoms = dataset['ucomp'].mean('lon') - dataset['ucomp'].mean(('time','lon'))
        else:
            # u_anoms_time = dataset.temporal.departures(data_var = 'ucomp', freq='day', weighted=False)['ucomp'].mean('lon')
            u_anoms_time = anom_ds['ucomp_anom']

        season_list = [season_val for season_val in season_month_dict.keys()]

        all_time_season_list = season_list+['all_time']

        eof_ds = xr.Dataset()
        eof_ds.coords['time'] = ('time', u_anoms_time.time.values)
        eof_ds.coords['pfull'] = ('pfull', u_anoms_time.sel(pfull=pfull_selector).pfull.values)  
        eof_ds.coords['lat'] = ('lat', u_anoms_time.sel(lat=hemisphere_slice_dict['n']).lat.values)                

        for season_val in season_list:
            u_anoms_season = u_anoms_time.where(anom_ds['time'].dt.month.isin(season_month_dict[season_val]), drop=True)
            eof_ds.coords[f'time_{season_val}'] = (f'time_{season_val}', u_anoms_season.time.values)

        eof_ds.coords['eof_num'] = ('eof_num', np.arange(n_eofs))

        ucomp_solver_dict = {}
        ucomp_500_solver_dict = {}        
        ucomp_va_solver_dict = {}        
        for season_val in all_time_season_list:
            ucomp_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_va_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_500_solver_dict[season_val] = {'n':{}, 's':{}}

        logging.info('about to propagate nans to all fields')
        if propagate_all_nans:
            anom_ds = propagate_missing_data_to_all_vars(anom_ds, eof_vars)

        for eof_var in eof_vars:

            logging.info('loading anomalies from anom_ds')

            var_anoms = anom_ds[f'{eof_var}_anom'].load()
            orig_var = anom_ds[f'{eof_var}_orig'].load()           

            for hemisphere in ['n','s']:

                for time_frame in all_time_season_list:
                    var_anoms_hem = var_anoms.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        var_anoms_hem = var_anoms_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True) 
                        time_dim_name = f'time_{time_frame}'
                    else:
                        time_dim_name = 'time'

                    orig_var_hem = orig_var.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        orig_var_hem = orig_var_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True)
                        
                    orig_var_hem = orig_var_hem.mean('time')   
                    
                    va_var_anoms_hem = vert_integrate(var_anoms_hem, pressure_weighted=True)

    return var_anoms_hem, va_var_anoms_hem, anom_ds

orig, va, anom_ds = obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset_miroc, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds_miroc, propagate_all_nans=True, level_subset=[250.,500.,850.])
anom_ds

2026-04-15 11:47:31,929 [INFO]: 2787009730.py(obtain_orig_var_hem:32) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-15 11:47:31,929 [INFO]: 2787009730.py(obtain_orig_var_hem:32) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-15 11:47:31,930 [INFO]: 2787009730.py(obtain_orig_var_hem:34) >> calculating anomalies
2026-04-15 11:47:31,930 [INFO]: 2787009730.py(obtain_orig_var_hem:34) >> calculating anomalies


2026-04-15 11:47:32,145 [INFO]: 2787009730.py(obtain_orig_var_hem:64) >> about to propagate nans to all fields
2026-04-15 11:47:32,145 [INFO]: 2787009730.py(obtain_orig_var_hem:64) >> about to propagate nans to all fields
2026-04-15 11:47:34,943 [INFO]: SIT_eddy_feedback_functions.py(propagate_missing_data_to_all_vars:660) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 3500356 nans
2026-04-15 11:47:34,943 [INFO]: SIT_eddy_feedback_functions.py(propagate_missing_data_to_all_vars:660) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 3500356 nans
2026-04-15 11:47:34,945 [INFO]: 2787009730.py(obtain_orig_var_hem:70) >> loading anomalies from anom_ds
2026-04-15 11:47:34,945 [INFO]: 2787009730.py(obtain_orig_var_hem:70) >> loading anomalies from anom_ds
2026-04-15 11:47:35,046 [INFO]: 2787009730.py(obtain_orig_var_hem:70) >> loading anomalies from an

<xarray.Dataset> Size: 606MB
Dimensions:           (pfull: 6, lat: 128, time: 13141)
Coordinates:
  * pfull             (pfull) float64 48B 925.0 850.0 700.0 600.0 500.0 250.0
  * lat               (lat) float64 1kB -88.93 -87.54 -86.14 ... 87.54 88.93
  * time              (time) datetime64[ns] 105kB 1979-01-01 ... 2015-01-01
Data variables:
    ucomp_anom        (time, pfull, lat) float64 81MB nan nan ... -3.969 -1.49
    ucomp_orig        (time, pfull, lat) float32 40MB nan nan ... -3.069 -1.209
    div1_QG_anom      (time, pfull, lat) float64 81MB nan nan nan ... 6.128 20.2
    div1_QG_orig      (time, pfull, lat) float64 81MB nan nan ... 9.775 22.14
    div1_QG_123_anom  (time, pfull, lat) float64 81MB nan nan ... 6.338 20.13
    div1_QG_123_orig  (time, pfull, lat) float64 81MB nan nan ... 10.01 22.05
    div1_QG_gt3_anom  (time, pfull, lat) float64 81MB nan nan ... 0.07251
    div1_QG_gt3_orig  (time, pfull, lat) float64 81MB nan nan ... 0.08862

In [5]:
print(anom_ds.ucomp_anom.size)
print(anom_ds.ucomp_anom.isnull().sum())

print((anom_ds.ucomp_anom.isnull().sum()/anom_ds.ucomp_anom.size)*100)

10092288
<xarray.DataArray 'ucomp_anom' ()> Size: 8B
array(3500356)
<xarray.DataArray 'ucomp_anom' ()> Size: 8B
array(34.68347316)


In [6]:
subset = anom_ds.ucomp_anom.sel(lat=slice(10., 80.)).sel(pfull=[250., 500., 850.], method='nearest')

# How many spatial points are contaminated (have ANY NaN across time)?
n_contaminated = subset.isnull().any('time').sum().values
n_total_spatial = subset.sizes['lat'] * subset.sizes['pfull']
print(f"Contaminated spatial points: {n_contaminated} / {n_total_spatial}")

# The killer question:
print(f"ALL spatial points have ≥1 NaN: {subset.isnull().any('time').all().values}")


Contaminated spatial points: 50 / 150
ALL spatial points have ≥1 NaN: False


In [7]:
def eof_calc_alt(data,lats):

    coslat = np.cos(np.deg2rad(lats)).clip(0., 1.)
    wgts = np.sqrt(coslat)[np.newaxis, np.newaxis, :]

    solver = Eof(data, weights=wgts, center=True)

    eofs = solver.eofsAsCovariance(neofs=3)
    pc1 = solver.pcs(npcs=3, pcscaling=1)

    variance_fractions = solver.varianceFraction(neigs=3)

    return eofs, pc1, variance_fractions, solver

eofs_va, pc1_va, variance_fractions_va, solver_va = eof_calc_alt(anom_ds.ucomp_anom.values, anom_ds.ucomp_anom.lat.values)

In [8]:
anom_ds.ucomp_anom

<xarray.DataArray 'ucomp_anom' (time: 13141, pfull: 6, lat: 128)> Size: 81MB
array([[[        nan,         nan,         nan, ...,  5.01253908,
          3.89007378,  1.91225491],
        [        nan,         nan,         nan, ...,  6.00891656,
          4.57874391,  2.09423224],
        [        nan,         nan,         nan, ...,  7.57974112,
          6.20801125,  3.07089793],
        [ 0.81567802,  1.26200832,  1.58738773, ...,  8.95356309,
          7.3077755 ,  3.61175932],
        [ 0.70197529,  1.25389552,  1.63365337, ..., 10.65564332,
          8.35907614,  3.95710717],
        [ 0.27479341,  1.38190813,  2.92345623, ...,  9.10436371,
          7.47764685,  4.54960376]],

       [[        nan,         nan,         nan, ...,  0.73301512,
          0.10355244, -0.56609643],
        [        nan,         nan,         nan, ...,  2.04012711,
          1.76152447,  0.63251517],
        [        nan,         nan,         nan, ...,  4.10628841,
          3.56371937,  1.83277907],
        [ 1.21121369,  1.48377101,  1.27961993, ...,  5.5804228 ,
...
         -4.32817497, -1.99122936],
        [ 1.28387496,  3.03579539,  3.99440423, ..., -7.94883975,
         -5.75995528, -2.38868623],
        [ 2.07364524,  4.55142857,  5.43313344, ..., -9.80391421,
         -6.59877111, -2.18058837],
        [ 1.61907264,  3.59016929,  4.56645212, ..., -5.33308846,
         -3.44887422, -1.72805249]],

       [[        nan,         nan,         nan, ..., -6.23212606,
         -3.70469951, -1.36017813],
        [        nan,         nan,         nan, ..., -6.26176339,
         -3.04011085, -0.80432638],
        [        nan,         nan,         nan, ..., -5.71534288,
         -3.29728188, -1.08110631],
        [ 1.08936067,  2.42703313,  3.34244828, ..., -6.95969642,
         -4.60920333, -1.2681763 ],
        [ 3.17323417,  5.33942425,  5.26415202, ..., -7.66305747,
         -5.53543318, -1.69766058],
        [ 1.8188664 ,  4.04208489,  4.56864886, ..., -5.78208801,
         -3.96930359, -1.49031943]]])
Coordinates:
  * pfull    (pfull) float64 48B 925.0 850.0 700.0 600.0 500.0 250.0
  * lat      (lat) float64 1kB -88.93 -87.54 -86.14 -84.74 ... 86.14 87.54 88.93
  * time     (time) datetime64[ns] 105kB 1979-01-01 1979-01-02 ... 2015-01-01
Attributes:
    operation:   temporal_avg
    mode:        departures
    freq:        day
    weighted:    True
    min_weight:  0.0